In [ ]:
from datetime import datetime
import re

class Account():
  _account_counter = 1000
  def __init__(self, account_holder: str, balance: float = 0):
    if balance < 0:
      raise ValueError("Начальный баланс не может быть отрицательным")

    self.account_type = ""
    self.holder = account_holder
    self.account_number = f"ACC-{Account._account_counter}"
    Account._account_counter += 1

    self._balance = float(balance)
    self.operations_history = []
  
  @staticmethod
  def _is_valid_holder(name: str) -> bool:
    if not isinstance(name, str):
        return False
    # «Имя Фамилия»: первая буква заглавная, остальные строчные кирриллица или латиница.
    pattern = r"^[А-ЯЁA-Z][а-яёa-z]+ [А-ЯЁA-Z][а-яёa-z]+$"
    return re.fullmatch(pattern, name) is not None

  def _add_operation(self, op_type: str, amount: float, status: str):
    self.operations_history.append({
      "type": op_type,
      "amount": amount,
      "datetime": datetime.now(),
      "balance": self._balance,
      "status": status
    })
  
  def deposit(self, amount: float):
    if amount < 0:
      raise ValueError("Сумма пополнения не может быть отрицательной")
    self.balance += amount
    self._add_operation(self, "deposit", "success")
    pass

  def withdraw(self, amount: float):
    if amount <= 0:
      raise ValueError("Сумма снятия должна быть положительной")
    if amount > self._balance:
      self._add_operation("withdraw", amount, "fail")
      return False
    self._balance -= amount
    self._add_operation("withdraw", amount, "success")
    return True
  
  def get_balance(self):
    return self.balance
  
  def get_history(self):
    return [operation.copy() for operation in self.operations_history]

  def analyze_operations(
    self,
    n: int = 5,
    sort_by: str = "amount",        # "amount" | "datetime" | "both"
    reverse: bool = True           # True — по убыванию, False — по возрастанию
):

    if n <= 0:
        return []

    if sort_by == "amount":
        key = lambda op: op["amount"]
    elif sort_by == "datetime":
        key = lambda op: op["datetime"]
    elif sort_by == "both":
        key = lambda op: (op["amount"], op["datetime"])
    else:
        raise ValueError("sort_by должен быть 'amount', 'datetime' или 'both'")

    ops = sorted(self.operations_history, key=key, reverse=reverse)
    result = ops[:n]

    for op in result:
        print(
            f"{op['datetime']:%Y-%m-%d %H:%M:%S} | "
            f"{op['type']:8} | "
            f"{op['amount']:.2f} | "
            f"баланс: {op['balance']:.2f} | "
            f"{op['status']}"
        )
    return result


class CheckingAccount(Account):
    account_type = "Checking"


class SavingsAccount(Account):
  account_type = "Savings"

  def apply_interest(self, rate: float):
    # rate is in percentage
    if rate < 0:
      raise ValueError("Процентная ставка не может быть отрицательной")
    
    interest = self._balance * rate / 100
    if interest == 0:
      return 0.0

    return interest
  
  def withdraw(self, amount):
    if amount <= 0:
      raise ValueError("Сумма снятия должна быть положительной")
    
    limit = self.get_balance() * 0.5
    if limit < amount:
      self._add_operation("withdraw", amount, "fail")
      return False
    
    self._balance -= amount
    self._add_operation("withdraw", amount, "success")
    return True
  
  
import unittest

class TestAccount(unittest.TestCase):

    def setUp(self):
        # Сбрасываем счётчик перед каждым тестом, чтобы номера были предсказуемы
        Account._account_counter = 1000
        
    def test_init_negative_balance_raises(self):
      with self.assertRaises(ValueError):
        Account("Иван Петров", -10)

    def test_account_number_format(self):
        acc = Account("Иван Петров")
        self.assertRegex(acc.account_number, r"^ACC-\d{4}$")